In [ ]:
# STEP 1 - Install packages
# Objective: install and load openxlsx (used to write the Excel workbook)
# Input:     none
# Output:    openxlsx loaded in the session
install.packages("openxlsx", quiet = TRUE)
library(openxlsx)

In [ ]:
# STEP 2 - Locate and verify the input files
# Objective: find the 5 students' CARD tables in /content and check that each one exists
# Input:     5 uploaded files in /content named CARD_Shahad.xlsx, CARD_Fulwah.xlsx,
#            CARD_Lamar.csv, CARD_Suhair.xlsx, CARD_Shaden.csv
# Output:    list f with the 5 file paths + a TRUE/FALSE existence check
dir  <- "/content"
allf <- list.files(dir, full.names = TRUE, pattern = "\\.(xlsx|csv)$", ignore.case = TRUE)
pick <- function(p) {
  x <- allf[grepl(p, basename(allf), ignore.case = TRUE)]
  if (length(x) != 1) stop("Expected exactly one file for: ", p, " (found ", length(x), ")")
  x
}
f <- list(Shahad = pick("^CARD_Shahad\\."), Fulwah = pick("^CARD_Fulwah\\."),
          Lamar  = pick("^CARD_Lamar\\."),  Suhair = pick("^CARD_Suhair\\."),
          Shaden = pick("^CARD_Shaden\\."))
order_students <- c("Shahad", "Fulwah", "Lamar", "Suhair", "Shaden")
print(unlist(f))

# Verification: every expected file must exist
check_files <- data.frame(student = names(f), file = basename(unlist(f)), exists = file.exists(unlist(f)), row.names = NULL)
print(check_files)
stopifnot(all(check_files$exists))

In [ ]:
# STEP 3 - Read the tables as they are
# Objective: read each student's table unchanged, drop empty rows and Fulwah's colour-key rows
# Input:     the 5 files from Step 2
# Output:    list raw (5 data frames) + number of gene rows per student
rd <- function(path) {
  if (grepl("\\.csv$", path, ignore.case = TRUE)) {
    df <- read.csv(path, check.names = FALSE, na.strings = "", colClasses = "character",
                   fileEncoding = "UTF-8-SIG", stringsAsFactors = FALSE)
  } else {
    df <- read.xlsx(path, sheet = 1, check.names = FALSE)
  }
  df[] <- lapply(df, as.character)
  m <- as.matrix(df)
  keep <- rowSums(!is.na(m) & trimws(m) != "") > 0
  df[keep, , drop = FALSE]
}
raw <- lapply(f[order_students], rd)

# remove the colour-key rows under Fulwah's table
fg <- raw$Fulwah[[1]]
raw$Fulwah <- raw$Fulwah[!grepl("^(colour key|green|yellow)", trimws(fg), ignore.case = TRUE), , drop = FALSE]

print(sapply(raw, nrow))

In [ ]:
# STEP 4 - Define the standardising functions
# Objective: map every student's columns to the 7 Merged columns, unify ARO ids and drug-class separators, flag exact matches
# Input:     none (definitions only)
# Output:    functions normalise(), is_green(), clean(), to_aro(), fix_drug() (no printed output)
maps <- list(
  Shahad = c(gene="amrfinderplus gene", fam="card gene / family", drug="drug class",
             mech="resistance mechanism", aro="card aro / url", mt="match status", notes="notes"),
  Fulwah = c(gene="amrfinderplus gene", fam="card gene / family", drug="drug class",
             mech="resistance mechanism", aro="card aro / url", mt="match type", notes="notes"),
  Shaden = c(gene="amrfinderplus gene", fam="card gene / family", drug="drug class",
             mech="resistance mechanism", aro="card aro / url", mt="match status", notes="notes"),
  Lamar  = c(gene="gene", fam="amr gene family", drug="drug class",
             mech="resistance mechanism", aro="card aro", mt="match status", notes=NA),
  Suhair = c(gene="amrfinderplus gene", fam="crrd gene /family", drug="drug class",
             mech="resistance mechanism", aro="card aro /url", mt=NA, notes="notes")
)

clean <- function(x, drop_placeholder = TRUE) {
  x <- trimws(gsub(" +", " ", gsub("[\t\r\n]+", " ", as.character(x))))
  x[x == ""] <- NA
  if (drop_placeholder) x[x %in% c("N/A", "n/a", "NA", "Not found in CARD")] <- NA
  x
}
fix_drug <- function(x) vapply(x, function(s) {
  if (is.na(s)) return(NA_character_)
  p <- trimws(unlist(strsplit(s, "[,;]")))
  paste(p[p != ""], collapse = "; ")
}, character(1), USE.NAMES = FALSE)
to_aro <- function(x) {
  x <- as.character(x)
  ifelse(grepl("[0-9]{4,}", x), paste0("ARO:", sub(".*?([0-9]{4,}).*", "\\1", x, perl = TRUE)), NA)
}
is_green <- function(mt) {
  s <- tolower(trimws(mt)); s[is.na(s)] <- ""
  s %in% c("exact name match", "exact match", "aro matched (status not recorded)") |
    startsWith(s, "exact-name candidate")
}

normalise <- function(df, who) {
  names(df) <- tolower(trimws(gsub("\\s+", " ", names(df))))
  m <- maps[[who]]
  col <- function(k) {
    nm <- m[[k]]
    if (is.na(nm)) return(rep(NA_character_, nrow(df)))
    if (!nm %in% names(df)) stop(who, ": Column not found: '", nm, "'. Available columns: ", paste(names(df), collapse=" | "))
    df[[nm]]
  }
  aro <- col("aro")
  if (who == "Lamar" && "card url" %in% names(df)) {
    aro <- ifelse(is.na(clean(aro)), df[["card url"]], aro)
  }
  out <- data.frame(
    `AMRFinderPlus Gene`   = clean(col("gene")),
    `CARD Gene / Family`   = clean(col("fam")),
    `Drug Class`           = fix_drug(clean(col("drug"))),
    `Resistance Mechanism` = clean(col("mech")),
    `CARD ARO / URL`       = to_aro(clean(aro)),
    `Match type`           = clean(col("mt"), drop_placeholder = FALSE),
    `Notes`                = clean(col("notes"), drop_placeholder = FALSE),
    check.names = FALSE, stringsAsFactors = FALSE)
  if (who == "Suhair") {
    need_review <- !is.na(out$Notes) | is.na(out$`CARD ARO / URL`)
    out$`Match type` <- ifelse(need_review, "Requires review", "ARO matched (status not recorded)")
  }
  out
}

In [ ]:
# STEP 5 - Merge and check
# Objective: stack the 5 tables into one, sort by gene, and check the result
# Input:     raw (Step 3) and the functions (Step 4)
# Output:    data frame merged (128 genes expected) + counts of green / uncoloured rows
merged <- do.call(rbind, lapply(order_students, function(w) normalise(raw[[w]], w)))
merged$green <- is_green(merged$`Match type`)
merged <- merged[order(merged$`AMRFinderPlus Gene`, method = "radix"), ]
rownames(merged) <- NULL

cat("Genes:", nrow(merged), "| duplicated:", sum(duplicated(merged$`AMRFinderPlus Gene`)), "\n")
if (nrow(merged) != 128 || anyDuplicated(merged$`AMRFinderPlus Gene`)) warning("Expected 128 unique genes!")
ref <- list.files(dir, pattern = "presence", full.names = TRUE)
if (length(ref) == 1) {
  g <- read.csv(ref, check.names = FALSE)[[1]]
  cat("Matches heatmap gene list:", setequal(g, merged$`AMRFinderPlus Gene`), "\n")
}
cat("Green:", sum(merged$green), "| Uncoloured (needs review):", sum(!merged$green), "\n")
head(merged[, 1:6], 10)

In [ ]:
# STEP 6 - Write the Excel workbook
# Objective: build CARD_merged.xlsx with the Merged sheet (green = exact match) + the 5 original sheets uncoloured
# Input:     merged (Step 5) and raw (Step 3)
# Output:    /content/CARD_merged.xlsx (6 sheets) + a check that the file was created
cols  <- setdiff(names(merged), "green")
hdr   <- createStyle(textDecoration = "bold", border = "bottom", valign = "top", wrapText = TRUE)
body  <- createStyle(wrapText = TRUE, valign = "top")
green <- createStyle(fgFill = "#C6EFCE")

wb <- createWorkbook()
addWorksheet(wb, "Merged")
writeData(wb, "Merged", merged[, cols], headerStyle = hdr)
n <- nrow(merged)
addStyle(wb, "Merged", body, rows = 2:(n + 1), cols = 1:7, gridExpand = TRUE)
gr <- which(merged$green) + 1
if (length(gr)) addStyle(wb, "Merged", green, rows = gr, cols = 1:7, gridExpand = TRUE, stack = TRUE)
setColWidths(wb, "Merged", cols = 1:7, widths = c(22, 28, 40, 30, 16, 34, 50))
freezePane(wb, "Merged", firstRow = TRUE)
writeData(wb, "Merged", "Key:", startRow = n + 3, startCol = 1)
writeData(wb, "Merged", "Green = exact name match found in CARD.", startRow = n + 3, startCol = 2)
writeData(wb, "Merged", "No colour = needs review (notation variant, family-level, unresolved, or not found in CARD).",
          startRow = n + 4, startCol = 2)

for (w in order_students) {
  sh <- sprintf("%s (%d genes)", w, nrow(raw[[w]]))
  addWorksheet(wb, sh)
  writeData(wb, sh, raw[[w]], headerStyle = createStyle(textDecoration = "bold"))
  setColWidths(wb, sh, cols = seq_len(ncol(raw[[w]])), widths = "auto")
}

out <- file.path(dir, "CARD_merged.xlsx")
saveWorkbook(wb, out, overwrite = TRUE)
cat("Saved:", out, "\n")
print(names(wb))

# Verification: output file exists and has the 6 expected sheets
stopifnot(file.exists(out))
print(getSheetNames(out))
stopifnot(length(getSheetNames(out)) == 6)